In [1]:
import sys
sys.path.append('/home/jovyan/work/Similarity-Aware-Label-Smoothing')

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
import pandas as pd
from models import CifarResNet18, CifarDenseNet121, TinyResNet34, TinyDenseNet121
from metrics import calibration_errors, nll_loss
import random
import os
from scipy.stats import spearmanr, wilcoxon



## Hyperparams

In [3]:
seed = 0
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [4]:
dataset = "cifar100"
batch_size = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lr = 0.1
epochs = 200
num_classes = 100
epsilon = 0.1
K = 1

## Training Utils

In [5]:
def accuracy(model, loader, k = (1, 5)):
    model.eval()
    correct = {key:0 for key in k}
    total = 0

    maxk = max(k)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)

            _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)

            for key in k:
                correct[key] += (pred[:, :key] == y.view(-1, 1)).any(dim=1).sum().item()
            total += y.size(0)

    return {key: correct[key] / total for key in k}

### Label Smoothing

In [6]:
path = f"Similarity-Aware-Label-Smoothing/scores/4_cifar100_latent_distances.xlsx"
dists = torch.tensor(pd.read_excel(io=path, sheet_name="centroid_distance").iloc[:, 1:].to_numpy(dtype=np.float32), dtype=torch.float32).to(device)

def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

def scores(y, k=K, tau=1.2):
    class_dists = dists[y]

    mask = torch.eye(class_dists.shape[1], device=device)[y]
    class_dists = class_dists.masked_fill(mask.bool(), float('inf'))
    sims = F.softmax(-class_dists / tau, dim=1)
    sims.scatter_(1, y.unsqueeze(1), 0.0)

    topk_values, topk_indices = torch.topk(sims, k, dim=1)
    result = torch.zeros_like(sims)
    result.scatter_(1, topk_indices, topk_values)

    result = result / (result.sum(dim=1, keepdim=True))

    return result

def similarity_aware_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return (1 - epsilon) * y_onehot + epsilon * scores(y)


## Training Loop

In [7]:
def train(model, train_loader, test_loader, classwise_target, optimizer=None, scheduler=None, epochs=10):

    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            targets = classwise_target[y]

            optimizer.zero_grad()
            logits = model(x)
            loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        test_acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")


## RUN Experiments

In [8]:
classwise_target = uniform_smooth_labels(torch.arange(end=num_classes).to(device), num_classes=num_classes, epsilon=epsilon)
print(classwise_target[0])
print(classwise_target.shape)

tensor([0.9000, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010,
        0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010,
        0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010,
        0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010,
        0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010,
        0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010,
        0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010,
        0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010,
        0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010,
        0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010,
        0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010, 0.0010,
        0.0010], device='cuda:0')
torch.Size([100, 100])


In [9]:
train_loader, test_loader = get_data_loaders(dataset=dataset)

In [ ]:
model = CifarDenseNet121(num_classes=num_classes).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4, nesterov=True)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, total_iters=5
        ),
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=195
        )
    ],
    milestones=[5]
)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    classwise_target=classwise_target,
    optimizer=optimizer,
    scheduler=scheduler,
    epochs=epochs,
)

Epoch 1/200 | Loss: 4.0324 | Test Acc: 0.1667 | Top-5 Test Acc: 0.4211
Epoch 2/200 | Loss: 3.6567 | Test Acc: 0.2134 | Top-5 Test Acc: 0.4954
Epoch 3/200 | Loss: 3.3676 | Test Acc: 0.2757 | Top-5 Test Acc: 0.5738
Epoch 4/200 | Loss: 3.1093 | Test Acc: 0.3270 | Top-5 Test Acc: 0.6379
Epoch 41/200 | Loss: 2.0727 | Test Acc: 0.4989 | Top-5 Test Acc: 0.7907
Epoch 42/200 | Loss: 2.0783 | Test Acc: 0.4788 | Top-5 Test Acc: 0.7681
Epoch 43/200 | Loss: 2.0572 | Test Acc: 0.5132 | Top-5 Test Acc: 0.8038
Epoch 44/200 | Loss: 2.0520 | Test Acc: 0.5354 | Top-5 Test Acc: 0.8200
Epoch 45/200 | Loss: 2.0447 | Test Acc: 0.5300 | Top-5 Test Acc: 0.8176
Epoch 46/200 | Loss: 2.0477 | Test Acc: 0.5070 | Top-5 Test Acc: 0.7957
Epoch 47/200 | Loss: 2.0386 | Test Acc: 0.4978 | Top-5 Test Acc: 0.7895
Epoch 48/200 | Loss: 2.0364 | Test Acc: 0.5120 | Top-5 Test Acc: 0.8055
Epoch 49/200 | Loss: 2.0303 | Test Acc: 0.5115 | Top-5 Test Acc: 0.7933
Epoch 50/200 | Loss: 2.0234 | Test Acc: 0.4989 | Top-5 Test Acc: 0.7

/opt/conda/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:209: UserWarning: The epoch parameter in `scheduler.step()` was not necessary and is being deprecated where possible. Please use `scheduler.step()` to step the scheduler. During the deprecation, if epoch is different from None, the closed form is used instead of the new chainable form, where available. Please open an issue if you are unable to replicate your use case: https://github.com/pytorch/pytorch/issues/new/choose.
  warnings.warn(EPOCH_DEPRECATION_WARNING, UserWarning)


Epoch 5/200 | Loss: 2.9104 | Test Acc: 0.3697 | Top-5 Test Acc: 0.6864
Epoch 6/200 | Loss: 2.7851 | Test Acc: 0.3737 | Top-5 Test Acc: 0.7007
Epoch 7/200 | Loss: 2.6436 | Test Acc: 0.4091 | Top-5 Test Acc: 0.7268
Epoch 8/200 | Loss: 2.5597 | Test Acc: 0.4147 | Top-5 Test Acc: 0.7275
Epoch 9/200 | Loss: 2.5013 | Test Acc: 0.4268 | Top-5 Test Acc: 0.7396
Epoch 10/200 | Loss: 2.4528 | Test Acc: 0.4366 | Top-5 Test Acc: 0.7503
Epoch 11/200 | Loss: 2.4199 | Test Acc: 0.4501 | Top-5 Test Acc: 0.7546
Epoch 12/200 | Loss: 2.3883 | Test Acc: 0.4590 | Top-5 Test Acc: 0.7723
Epoch 13/200 | Loss: 2.3618 | Test Acc: 0.4480 | Top-5 Test Acc: 0.7565
Epoch 14/200 | Loss: 2.3411 | Test Acc: 0.4565 | Top-5 Test Acc: 0.7600
Epoch 15/200 | Loss: 2.3191 | Test Acc: 0.4748 | Top-5 Test Acc: 0.7765
Epoch 16/200 | Loss: 2.2994 | Test Acc: 0.4530 | Top-5 Test Acc: 0.7657
Epoch 17/200 | Loss: 2.2845 | Test Acc: 0.4771 | Top-5 Test Acc: 0.7757
Epoch 18/200 | Loss: 2.2650 | Test Acc: 0.4735 | Top-5 Test Acc: 0.77

In [ ]:
logits_list = []
labels_list = []

model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)

        logits_list.append(logits)
        labels_list.append(y)

logits_all = torch.cat(logits_list, dim=0)
labels_all = torch.cat(labels_list, dim=0)
print()
print("ECE:", calibration_errors(logits_all, labels_all))
print("NLL:", nll_loss(logits_all, labels_all))
test_acc = accuracy(model, test_loader)
print(f"Top-1 Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")
print()



ECE: (0.07628405094146729, 0.2510398030281067)
NLL: 1.5147866010665894
Top-1 Test Acc: 0.6712 | Top-5 Test Acc: 0.8635



In [ ]:
PATH = f"vae8_{'0'+str(epsilon).removeprefix("0.")}_{K}_{seed}.pth"
torch.save(model.state_dict(), PATH)